# Signal Processing: Filters

## Environment set up

Change the working directory to be able to work with the source-code of this repository.

In [ ]:
import os
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().parents[0]
os.chdir(WORKING_DIRECTORY)

## Imports

In [ ]:
from src.read import read_nasa_vibration_files_in_directory
from src.signals.processing import Signal, band_pass_filter, low_pass_filter, high_pass_filter, envelope, power_spectrum, process_signal
from typing import Optional
from src.signals import calculations
import matplotlib.pyplot as plt
import numpy as np
from loguru import logger
import matplotlib.dates as mdates
import polars as pl
import plotly.express as px
from datetime import datetime

## Inputs

The inputs have been obtained from the NASA bearings documentation.

The following cell displays the data path for each test and the name of their columns:

In [ ]:
DATA_INPUTS_PER_TEST = {
    '1st_test': {'data_path': 'data/nasa_ims_bearing_dataset/1st_test',
                  'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4',
                                   'channel_5', 'channel_6', 'channel_7', 'channel_8']},
    '2nd_test': {'data_path': 'data/nasa_ims_bearing_dataset/2nd_test',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']},
    '3rd_test': {'data_path': 'data/nasa_ims_bearing_dataset/3rd_test/4th_test/txt',
                 'column_names': ['channel_1', 'channel_2', 'channel_3', 'channel_4']}
          }

As each test has a different set up of sensors or channels per bearing, the following cell describes them:

In [ ]:
BEARING_CHANNEL_MAPPING = {
    '1st_test': {'bearing_1': ['channel_1', 'channel_2'],
                 'bearing_2': ['channel_3', 'channel_4'],
                 'bearing_3': ['channel_5', 'channel_6'],
                 'bearing_4': ['channel_7', 'channel_8']},
    '2nd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']},
    '3rd_test': {'bearing_1': ['channel_1'],
                 'bearing_2': ['channel_2'],
                 'bearing_3': ['channel_3'],
                 'bearing_4': ['channel_4']}                 
}

Next, the faulty bearings are defined per test:

In [ ]:
FAULTY_BEARINGS_PER_TEST = {
    '1st_test': {'bearing_3': 'bearing_inner_race',
                 'bearing_4': 'bearing_roller'
                 },
    '2nd_test': {'bearing_1': 'bearing_outer_race'},
    '3rd_test': {'bearing_3': 'bearing_outer_race'}
}

As final inputs, the following parameters are needed to read properly the vibration signals. In addition, an acceptable sensor range is defined to avoid faulty channel signals:

In [ ]:
SAMPLING_FREQUENCY = 20000
MEASUREMENT_DURATION_IN_SECONDS = 1
ACCEPTABLE_SENSOR_RANGE = 0.01

## Read the Data

In [ ]:
complete_data_path_per_test = {}

for test, inputs_per_test in DATA_INPUTS_PER_TEST.items():
    for key, values in inputs_per_test.items():
        data_path = inputs_per_test['data_path']
        complete_path = WORKING_DIRECTORY.joinpath(data_path)
        complete_data_path_per_test[test] = complete_path


In [ ]:
signal_resolution = calculations.resolution(sampling_frequency=SAMPLING_FREQUENCY)

df_list_per_test = {}
for test, file_path in complete_data_path_per_test.items():
    logger.info(f'test: {test}')
    column_names = DATA_INPUTS_PER_TEST[test]['column_names']
    df_list = read_nasa_vibration_files_in_directory(files_path=file_path, sensors=column_names,
                                                     signal_resolution=signal_resolution,
                                                     acceptable_sensor_range=ACCEPTABLE_SENSOR_RANGE)
    df_list_sorted = sorted(df_list, key=lambda df: datetime.strptime(df['file_name'][0], '%Y.%m.%d.%H.%M.%S')) 
    df_list_per_test[test] = df_list_sorted

## Sample Signal

In [ ]:
TEST= '2nd_test'
CHANNEL = 'channel_1'

sample_signal_df= df_list_per_test[TEST][-1]
x = sample_signal_df['measurement_time_in_seconds'].to_numpy()
y = sample_signal_df[CHANNEL].to_numpy()

sample_signal = Signal(x=x, y=y)
print(f'sample_signal: {sample_signal}')


In [ ]:
measurement_recording_date_time = sample_signal.x[-1]

measurement_duration = sample_signal.x[-1]
signal_resolution = calculations.resolution(sampling_frequency=SAMPLING_FREQUENCY)
signal_resolution_manual_calculation = sample_signal.x[1] - sample_signal.x[0]


print(f'measurement_duration: {measurement_duration} s')
print(f'signal_resolution: {signal_resolution} s')
print(f'signal_resolution_manual_calculation: {signal_resolution_manual_calculation} s')


In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(sample_signal.x, sample_signal.y)
plt.suptitle(f'Sample Raw Signal: {TEST} - {CHANNEL} - {measurement_recording_date_time}')
plt.title(f'Sampling frequency: {SAMPLING_FREQUENCY} Hz - Measurement duration: {measurement_duration:.4f} s')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (g)')
plt.xticks(np.arange(0, sample_signal.x[-1]+0.05, 0.05))
plt.grid()
plt.show()

## Plotting Function

The following function has been created to facilitate the analysis:

In [ ]:
def plot_signals_and_spectra(
    sample_signal: Signal,
    filtered_signal: Signal,
    sample_power_spectrum_signal: Signal,
    filtered_power_spectrum_signal: Signal,
    test: str,
    channel: str,
    measurement_recording_date_time: str,
    sampling_frequency: float,
    measurement_duration: float,
    low_cutoff_frequency: Optional[float] = None,
    high_cutoff_frequency: Optional[float] = None,
    filter_label: str = 'filtered signal',
    time_xtick_step: Optional[float] = 0.05,
    freq_xtick_step: Optional[float] = 500,
) -> None:
    """Create time-domain (overlaid) and frequency-domain (stacked, shared x-axis) plots.

    Determines plot titles from which cutoff frequencies are provided:
    - low_cutoff_frequency only -> Low-pass
    - high_cutoff_frequency only -> High-pass
    - both provided -> Band-pass

    Parameters
    - sample_signal, filtered_signal: `Signal` objects with `.x` and `.y` (numpy arrays).
    - sample_power_spectrum_signal, filtered_power_spectrum_signal: `Signal` objects with `.x` and `.y`.
    - test, channel, measurement_recording_date_time: strings used in titles.
    - sampling_frequency, measurement_duration, low_cutoff_frequency, high_cutoff_frequency: numeric metadata.
    - filter_label: legend label for the filtered trace.
    - time_xtick_step, freq_xtick_step: tick spacing for x-axes.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    # Decide filter type and build subtitle pieces
    if low_cutoff_frequency is not None and high_cutoff_frequency is not None:
        filter_type = 'Band-pass'
        cutoff_info = f'Low Cutoff Frequency: {low_cutoff_frequency} Hz - High Cutoff Frequency: {high_cutoff_frequency} Hz'
    elif low_cutoff_frequency is not None:
        filter_type = 'Low-pass'
        cutoff_info = f'Cutoff Frequency: {low_cutoff_frequency} Hz'
    elif high_cutoff_frequency is not None:
        filter_type = 'High-pass'
        cutoff_info = f'Cutoff Frequency: {high_cutoff_frequency} Hz'
    else:
        filter_type = 'Filtered'
        cutoff_info = ''

    # Time-domain (overlaid)
    plt.figure(figsize=(15, 5))
    plt.plot(sample_signal.x, sample_signal.y, label='raw signal')
    plt.plot(filtered_signal.x, filtered_signal.y, label=filter_label)
    plt.suptitle(f'Sample Raw Signal: {test} - {channel} - {measurement_recording_date_time}')
    plt.title(
        f'Sampling frequency: {sampling_frequency} Hz - Measurement duration: {measurement_duration:.4f} s'
        + (f' - {cutoff_info}' if cutoff_info else '')
    )
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (g)')
    if time_xtick_step and len(filtered_signal.x):
        try:
            end = float(filtered_signal.x[-1])
            step = float(time_xtick_step)
            plt.xticks(np.arange(0, end + step, step))
        except Exception:
            pass
    plt.legend()
    plt.grid()
    plt.show()

    # Frequency-domain (two stacked subplots sharing x-axis)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 5), sharex=True)

    ax1.plot(sample_power_spectrum_signal.x, sample_power_spectrum_signal.y, label='raw signal')
    ax1.set_ylabel('Amplitude (g)')
    ax1.grid()
    ax1.legend()

    ax2.plot(filtered_power_spectrum_signal.x, filtered_power_spectrum_signal.y, label=filter_label, color='tab:orange')
    ax2.set_ylabel('Amplitude (g)')
    ax2.set_xlabel('Frequency (Hz)')
    ax2.grid()
    ax2.legend()

    plt.suptitle(
        f'Power Spectrum: {test} - {channel} - {measurement_recording_date_time}\n'
        f'Sampling frequency: {sampling_frequency} Hz - Measurement duration: {measurement_duration:.4f} s'
        + (f' - {cutoff_info}' if cutoff_info else '')
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if freq_xtick_step and len(sample_power_spectrum_signal.x):
        try:
            endf = float(sample_power_spectrum_signal.x[-1])
            stepf = float(freq_xtick_step)
            plt.xticks(np.arange(0, endf + stepf, stepf))
        except Exception:
            pass
    plt.show()


## Signal Processing

### Low-Pass Filter

A low-pass filter is a filter that passes signals with a frequency lower than a selected cutoff frequency and attenuates signals with frequencies higher than the cutoff frequency. The exact frequency response of the filter depends on the filter design. (Source: Wikipedia)

In [ ]:
LOW_PASS_CUTOFF_FREQUENCY = 2500

low_pass_filtered_signal = low_pass_filter(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY, cutoff_frequency=LOW_PASS_CUTOFF_FREQUENCY)

sample_power_spectrum_signal = power_spectrum(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY)
low_pass_filtered_power_spectrum_signal = power_spectrum(signal=low_pass_filtered_signal, sampling_frequency=SAMPLING_FREQUENCY)


plot_signals_and_spectra(sample_signal, low_pass_filtered_signal, sample_power_spectrum_signal, low_pass_filtered_power_spectrum_signal, 
                      TEST, CHANNEL, measurement_recording_date_time, SAMPLING_FREQUENCY, measurement_duration, 
                      low_cutoff_frequency=LOW_PASS_CUTOFF_FREQUENCY, filter_label='low pass filtered signal')

### High-Pass Filter

A high-pass filter (HPF) is an electronic filter that passes signals with a frequency higher than a certain cutoff frequency and attenuates signals with frequencies lower than the cutoff frequency. (Source: Wikipedia)

In [ ]:
HIGH_PASS_CUTOFF_FREQUENCY = 2500

high_pass_filtered_signal = high_pass_filter(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY, cutoff_frequency=HIGH_PASS_CUTOFF_FREQUENCY)

sample_power_spectrum_signal = power_spectrum(signal=sample_signal, sampling_frequency=SAMPLING_FREQUENCY)
high_pass_filtered_power_spectrum_signal = power_spectrum(signal=high_pass_filtered_signal, sampling_frequency=SAMPLING_FREQUENCY)


plot_signals_and_spectra(sample_signal, high_pass_filtered_signal, sample_power_spectrum_signal, high_pass_filtered_power_spectrum_signal,
                      TEST, CHANNEL, measurement_recording_date_time, SAMPLING_FREQUENCY, measurement_duration,
                      high_cutoff_frequency=HIGH_PASS_CUTOFF_FREQUENCY, filter_label='high pass filtered signal')

### Band-Pass-Filter

A band-pass filter or bandpass filter (BPF) is a device that passes frequencies within a certain range and rejects (attenuates) frequencies outside that range. (Source: Wikipedia)

In [ ]:
LOW_CUTOFF_FREQUENCY = 2500
HIGH_CUTOFF_FREQUENCY = 5000

high_pass_filtered_signal = band_pass_filter(signal=sample_signal, 
                                             sampling_frequency=SAMPLING_FREQUENCY, 
                                             low_cutoff_frequency=LOW_CUTOFF_FREQUENCY,
                                             high_cutoff_frequency=HIGH_CUTOFF_FREQUENCY)

sample_power_spectrum_signal = power_spectrum(signal=sample_signal, 
                                              sampling_frequency=SAMPLING_FREQUENCY)
band_pass_filtered_power_spectrum_signal = power_spectrum(signal=high_pass_filtered_signal, 
                                                          sampling_frequency=SAMPLING_FREQUENCY)


plt.figure(figsize=(15, 5))
plt.plot(sample_signal.x, sample_signal.y, label='raw signal')
plt.plot(high_pass_filtered_signal.x, high_pass_filtered_signal.y, label='high pass filtered signal')
plt.suptitle(f'Sample Raw Signal: {TEST} - {CHANNEL} - {measurement_recording_date_time}')
plt.title(f'Sampling frequency: {SAMPLING_FREQUENCY} Hz - Measurement duration: {measurement_duration:.4f} s - '  
          f'Low Cutoff Frequency: {LOW_CUTOFF_FREQUENCY} Hz - High Cutoff Frequency: {HIGH_CUTOFF_FREQUENCY} Hz')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (g)')
plt.xticks(np.arange(0, low_pass_filtered_signal.x[-1]+0.05, 0.05))
plt.legend()
plt.grid()
plt.show();


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 5), sharex=True)

ax1.plot(sample_power_spectrum_signal.x, sample_power_spectrum_signal.y, label='raw signal')
ax1.set_ylabel('Amplitude (g)')
ax1.grid()
ax1.legend()

ax2.plot(band_pass_filtered_power_spectrum_signal.x, band_pass_filtered_power_spectrum_signal.y, label='band pass filtered signal', color='tab:orange')
ax2.set_ylabel('Amplitude (g)')
ax2.set_xlabel('Frequency (Hz)')
ax2.grid()
ax2.legend()

plt.suptitle(f'Power Spectrum: {TEST} - {CHANNEL} - {measurement_recording_date_time}\nSampling frequency: {SAMPLING_FREQUENCY} Hz - Measurement duration: {measurement_duration:.4f} s - '  
          f'Low Cutoff Frequency: {LOW_CUTOFF_FREQUENCY} Hz - High Cutoff Frequency: {HIGH_CUTOFF_FREQUENCY} Hz')
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.xticks(np.arange(0, sample_power_spectrum_signal.x[-1]+500, 500))
plt.show();

In [ ]:
from typing import Optional

def plot_signals_and_spectra(
    sample_signal: Signal,
    filtered_signal: Signal,
    sample_power_spectrum_signal: Signal,
    filtered_power_spectrum_signal: Signal,
    test: str,
    channel: str,
    measurement_recording_date_time: str,
    sampling_frequency: float,
    measurement_duration: float,
    low_cutoff_frequency: Optional[float] = None,
    high_cutoff_frequency: Optional[float] = None,
    filter_label: str = 'filtered signal',
    time_xtick_step: Optional[float] = 0.05,
    freq_xtick_step: Optional[float] = 500,
) -> None:
    """Create time-domain (overlaid) and frequency-domain (stacked, shared x-axis) plots.

    Determines plot titles from which cutoff frequencies are provided:
    - low_cutoff_frequency only -> Low-pass
    - high_cutoff_frequency only -> High-pass
    - both provided -> Band-pass

    Parameters
    - sample_signal, filtered_signal: `Signal` objects with `.x` and `.y` (numpy arrays).
    - sample_power_spectrum_signal, filtered_power_spectrum_signal: `Signal` objects with `.x` and `.y`.
    - test, channel, measurement_recording_date_time: strings used in titles.
    - sampling_frequency, measurement_duration, low_cutoff_frequency, high_cutoff_frequency: numeric metadata.
    - filter_label: legend label for the filtered trace.
    - time_xtick_step, freq_xtick_step: tick spacing for x-axes.
    """
    import matplotlib.pyplot as plt
    import numpy as np

    # Decide filter type and build subtitle pieces
    if low_cutoff_frequency is not None and high_cutoff_frequency is not None:
        filter_type = 'Band-pass'
        cutoff_info = f'Low Cutoff Frequency: {low_cutoff_frequency} Hz - High Cutoff Frequency: {high_cutoff_frequency} Hz'
    elif low_cutoff_frequency is not None:
        filter_type = 'Low-pass'
        cutoff_info = f'Cutoff Frequency: {low_cutoff_frequency} Hz'
    elif high_cutoff_frequency is not None:
        filter_type = 'High-pass'
        cutoff_info = f'Cutoff Frequency: {high_cutoff_frequency} Hz'
    else:
        filter_type = 'Filtered'
        cutoff_info = ''

    # Time-domain (overlaid)
    plt.figure(figsize=(15, 5))
    plt.plot(sample_signal.x, sample_signal.y, label='raw signal')
    plt.plot(filtered_signal.x, filtered_signal.y, label=filter_label)
    plt.suptitle(f'Sample Raw Signal: {test} - {channel} - {measurement_recording_date_time}')
    plt.title(
        f'Sampling frequency: {sampling_frequency} Hz - Measurement duration: {measurement_duration:.4f} s'
        + (f' - {cutoff_info}' if cutoff_info else '')
    )
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude (g)')
    if time_xtick_step and len(filtered_signal.x):
        try:
            end = float(filtered_signal.x[-1])
            step = float(time_xtick_step)
            plt.xticks(np.arange(0, end + step, step))
        except Exception:
            pass
    plt.legend()
    plt.grid()
    plt.show()

    # Frequency-domain (two stacked subplots sharing x-axis)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 5), sharex=True)

    ax1.plot(sample_power_spectrum_signal.x, sample_power_spectrum_signal.y, label='raw signal')
    ax1.set_ylabel('Amplitude (g)')
    ax1.grid()
    ax1.legend()

    ax2.plot(filtered_power_spectrum_signal.x, filtered_power_spectrum_signal.y, label=filter_label, color='tab:orange')
    ax2.set_ylabel('Amplitude (g)')
    ax2.set_xlabel('Frequency (Hz)')
    ax2.grid()
    ax2.legend()

    plt.suptitle(
        f'Power Spectrum: {test} - {channel} - {measurement_recording_date_time}\n'
        f'Sampling frequency: {sampling_frequency} Hz - Measurement duration: {measurement_duration:.4f} s'
        + (f' - {cutoff_info}' if cutoff_info else '')
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if freq_xtick_step and len(sample_power_spectrum_signal.x):
        try:
            endf = float(sample_power_spectrum_signal.x[-1])
            stepf = float(freq_xtick_step)
            plt.xticks(np.arange(0, endf + stepf, stepf))
        except Exception:
            pass
    plt.show()
